# Biophysical Realism: pH Titration & Salt Bridges

This interactive lab demonstrates how `synth-pdb` models realistic physical chemistry. Proteins exist in dynamic, aqueous environments where the pH dictates the protonation state of titratable residues. The most critical of these near physiological pH is **Histidine**, which has a pKa of ~6.0.

In this notebook, we will:
1. Generate a synthetic peptide enriched in acidic (Aspartate, Glutamate) and basic (Histidine) residues.
2. Apply different pH values to observe Histidine protonation (`HIS` $\rightarrow$ `HIP`).
3. Visualize the physical formation of electrostatic **Salt Bridges** between these charged groups.


In [1]:
import io
import py3Dmol
import biotite.structure.io.pdb as pdb
from synth_pdb.generator import generate_pdb_content
from synth_pdb.biophysics import apply_ph_titration, find_salt_bridges, cap_termini

print("Libraries loaded successfully.")

Libraries loaded successfully.


## 1. Generating the Peptide
We'll synthesize an alpha helix sequence designed to bring basic and acidic residues into proximity: `EAAAAHAAAAE` (Glutamate, Histidine, Glutamate).

In [2]:
# Generate a synthetic alpha helix
sequence = "EAAAAHAAAAE"
raw_pdb = generate_pdb_content(sequence_str=sequence, conformation="alpha")

# Load into Biotite for biophysical processing
pdb_file = pdb.PDBFile.read(io.StringIO(raw_pdb))
structure = pdb_file.get_structure(model=1)

# Apply terminal caps (ACE/NME) to neutralize the artificial chain ends
structure = cap_termini(structure)

print(f"Generated peptide with {len(structure)} atoms.")

Generated peptide with 135 atoms.


## 2. Interactive pH Visualization

Run the cell below. A slider will appear allowing you to change the pH. 
Watch what happens to the central Histidine when the pH drops below 6.0!

In [3]:
import ipywidgets as widgets
from IPython.display import display


def visualize_ph(ph_value):
    # 1. Apply pH titration
    # This will change HIS to HIP (+1 charge) at pH < 6.0
    titrated_struct = structure.copy()
    titrated_struct = apply_ph_titration(titrated_struct, ph=ph_value)

    # 2. Find resulting Salt Bridges
    salt_bridges = find_salt_bridges(titrated_struct, cutoff=5.0)

    # Convert back to PDB format for viewing
    out_file = pdb.PDBFile()
    out_file.set_structure(titrated_struct)
    out_str = io.StringIO()
    out_file.write(out_str)
    pdb_data = out_str.getvalue()

    # 3. Setup 3D Viewer
    view = py3Dmol.view(width=800, height=400)
    view.addModel(pdb_data, "pdb")

    # Style: Grey backbone, colored sidechains for Acidic (Red) and Basic (Blue)
    view.setStyle({"cartoon": {"color": "lightgrey", "opacity": 0.8}})
    view.addStyle({"resn": ["GLU", "ASP"]}, {"stick": {"colorscheme": "redCarbon", "radius": 0.2}})
    view.addStyle(
        {"resn": ["HIS", "HIP", "HIE", "HID"]},
        {"stick": {"colorscheme": "blueCarbon", "radius": 0.2}},
    )

    # 4. Draw Salt Bridges as yellow dashed cylinders
    bridge_count = 0
    for bridge in salt_bridges:
        # We need to extract coordinates to draw a cylinder
        atom_a = titrated_struct[
            (titrated_struct.res_id == bridge["res_ia"])
            & (titrated_struct.atom_name == bridge["atom_a"])
        ][0]
        atom_b = titrated_struct[
            (titrated_struct.res_id == bridge["res_ib"])
            & (titrated_struct.atom_name == bridge["atom_b"])
        ][0]

        view.addCylinder(
            {
                "start": {"x": atom_a.coord[0], "y": atom_a.coord[1], "z": atom_a.coord[2]},
                "end": {"x": atom_b.coord[0], "y": atom_b.coord[1], "z": atom_b.coord[2]},
                "radius": 0.1,
                "color": "yellow",
                "dashed": True,
            }
        )
        bridge_count += 1

    view.zoomTo()
    view.show()

    print(f"--- pH: {ph_value} ---")
    print(f"Salt bridges detected: {bridge_count}")
    if bridge_count > 0:
        print("Notice the yellow cylinders representing electrostatic attraction!")


widgets.interact(
    visualize_ph,
    ph_value=widgets.FloatSlider(min=2.0, max=10.0, step=0.5, value=7.4, description="pH:"),
);

interactive(children=(FloatSlider(value=7.4, description='pH:', max=10.0, min=2.0, step=0.5), Output()), _dom_…